In [2]:
import matplotlib.pyplot as plt
import os
import numpy as np
from cffi import FFI
ffi = FFI()

ffi.cdef("""
    typedef struct {
        float* train_cat;
        float* test_cat;
        float* train_lion;
        float* test_lion;
        float* train_cheetah;
        float* test_cheetah;
        size_t epochs;
    } EvalHistoryFFI;

    EvalHistoryFFI* train_classifier_with_eval(
        void* classifier,
        const float* x, size_t rows, size_t cols,
        const size_t* y, size_t y_len,
        const float* x_test, size_t test_rows,
        const size_t* y_test, size_t y_test_len,
        size_t epochs
    );
""", override=True)

dll_path = os.path.abspath("../felidae_classifier/target/debug/felidae_classifier.dll")
lib = ffi.dlopen(dll_path)


epochs = 100

result = lib.train_classifier_with_eval(
    classifier,
    X_ptr, X.shape[0], X.shape[1],
    Y_ptr, Y.shape[0],
    X_test_ptr, X_test.shape[0],
    Y_test_ptr, Y_test.shape[0],
    epochs
)

def to_np(ptr, n):
    return np.frombuffer(ffi.buffer(ptr, n * ffi.sizeof("float")), dtype=np.float32)

train_cat = to_np(result.train_cat, epochs)
test_cat = to_np(result.test_cat, epochs)
train_lion = to_np(result.train_lion, epochs)
test_lion = to_np(result.test_lion, epochs)
train_cheetah = to_np(result.train_cheetah, epochs)
test_cheetah = to_np(result.test_cheetah, epochs)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, tr, te) in zip(axes, [
    ("Cat", train_cat, test_cat),
    ("Lion", train_lion, test_lion),
    ("Cheetah", train_cheetah, test_cheetah),
]):
    ax.plot(tr, label="Train")
    ax.plot(te, label="Test")
    ax.set_title(name)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.legend()
plt.tight_layout()
plt.show()

NameError: name 'classifier' is not defined